In [1]:
import pandas as pd
import numpy as np
import joblib
import time
from datetime import datetime

In [2]:
test_df = pd.read_csv("../data/processed/test_data.csv")
print("Test Data Shape:", test_df.shape)
test_df.head()

Test Data Shape: (200000, 48)


,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,zip_count_4w,velocity_6h,velocity_24h,...,housing_status_3,housing_status_4,housing_status_5,housing_status_6,source_1,device_os_1,device_os_2,device_os_3,device_os_4,fraud_bool
0,0.6,0.997643,34.0,61.0,20,0.027197,-0.891201,5431,6347.580973,6794.366547,...,0,0,0,0,0,1,0,0,0,0
1,0.1,0.488162,28.0,16.0,20,0.003582,-0.233423,3343,5680.515099,5532.075769,...,0,1,0,0,0,1,0,0,0,0
2,0.4,0.987888,34.0,169.0,40,0.003197,25.079727,654,1822.678744,3362.935558,...,0,0,0,0,0,0,1,0,0,0
3,0.1,0.860912,34.0,86.0,40,0.009014,-1.533378,2170,1013.218122,5812.376534,...,0,0,0,0,0,0,0,0,0,0
4,0.1,0.194331,34.0,92.0,30,8.851038,34.349159,1152,7354.861925,6334.257480,...,0,1,0,0,0,0,1,0,0,0


In [3]:
model = joblib.load("../models/random_forest.pkl")
print("Model Loaded Successfully")

Model Loaded Successfully


In [4]:
X_test = test_df.drop("fraud_bool", axis=1)
y_test = test_df["fraud_bool"]
print(X_test.shape)

(200000, 47)


In [5]:
#Creating Real-Time Transaction Simulator . This simulates incoming banking transactions.
def transaction_stream(data, n_transactions=10):
    for i in range(n_transactions):
        transaction = data.sample(1)
        yield transaction
        time.sleep(1)

In [6]:
#Fraud Prediction Function
def predict_transaction(transaction):
    probability = model.predict_proba(transaction)[0][1]
    prediction = model.predict(transaction)[0]
    if probability >= 0.75:
        risk = "High Risk"
    elif probability >= 0.40:
        risk = "Medium Risk"
    else:
        risk = "Low Risk"
    return prediction, probability, risk

In [7]:
#Run Real-Time Monitoring
alerts = []
for transaction in transaction_stream(X_test,10):
    prediction, probability, risk = predict_transaction(transaction)
    result = {
        "timestamp": datetime.now(),
        "fraud_prediction": prediction,
        "fraud_probability": probability,
        "risk_level": risk
    }
    alerts.append(result)
    print(result)

{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 25, 254067), 'fraud_prediction': np.int64(0), 'fraud_probability': np.float64(0.009508140752346855), 'risk_level': 'Low Risk'}
{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 26, 449009), 'fraud_prediction': np.int64(0), 'fraud_probability': np.float64(0.04865129912433006), 'risk_level': 'Low Risk'}
{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 27, 658591), 'fraud_prediction': np.int64(0), 'fraud_probability': np.float64(0.13510718766341287), 'risk_level': 'Low Risk'}
{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 28, 878751), 'fraud_prediction': np.int64(1), 'fraud_probability': np.float64(0.624430706983413), 'risk_level': 'Medium Risk'}
{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 30, 46518), 'fraud_prediction': np.int64(0), 'fraud_probability': np.float64(0.36151636282558086), 'risk_level': 'Low Risk'}
{'timestamp': datetime.datetime(2026, 7, 28, 23, 48, 31, 257463), 'fraud_prediction': np.int64(0), 'frau

In [8]:
#convert alerts into dataframe
monitoring_df = pd.DataFrame(alerts)
monitoring_df

,timestamp,fraud_prediction,fraud_probability,risk_level
0,2026-07-28 23:48:25.254067,0,0.009508,Low Risk
1,2026-07-28 23:48:26.449009,0,0.048651,Low Risk
2,2026-07-28 23:48:27.658591,0,0.135107,Low Risk
3,2026-07-28 23:48:28.878751,1,0.624431,Medium Risk
4,2026-07-28 23:48:30.046518,0,0.361516,Low Risk
5,2026-07-28 23:48:31.257463,0,0.277379,Low Risk
6,2026-07-28 23:48:32.411636,0,0.007750,Low Risk
7,2026-07-28 23:48:33.525390,0,0.033562,Low Risk
8,2026-07-28 23:48:34.759357,0,0.026014,Low Risk
9,2026-07-28 23:48:35.891671,0,0.060439,Low Risk


In [9]:
monitoring_df.to_csv("../data/processed/realtime_fraud_monitoring.csv",index=False)
print("Monitoring Results Saved Successfully")

Monitoring Results Saved Successfully


# Notebook 17 Summary

- Loaded the trained fraud detection model.
- Created a real-time transaction simulation environment.
- Generated fraud predictions for incoming transactions.
- Calculated fraud probability scores.
- Classified transactions into Low, Medium, and High risk categories.
- Generated fraud monitoring alerts.
- Saved real-time fraud monitoring results for further analysis.